In [ ]:
#import drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import ee
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-sample1234')

In [ ]:
import json

# Load the GeoJSON file from Google Drive
with open('/content/drive/MyDrive/GEE/RegionOfInterest.geojson') as f:
    geojson = json.load(f)

with open('/content/drive/MyDrive/GEE/SyedanWalan GeoJason.geojson') as f:
    geojson = json.load(f)

# Convert to Earth Engine Geometry
roi = ee.Feature(geojson['features'][0]).geometry()

In [ ]:
def clip_to_roi(image):
    return image.clip(roi).copyProperties(image, image.propertyNames())

def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

def set_date(img):
    date_str = img.date().format("YYYYMMdd")
    return img.set("date", date_str)
# Merge, sort, scale, and mask clouds

dataset = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .sort("system:time_start")
    .filterBounds(roi)
    #.filterDate("2023-11-17", "2024-04-23")
    .filterDate("2018-01-01", "2022-04-01")
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(mask_s2_clouds)
    .map(clip_to_roi)
    .map(set_date)
)

In [ ]:
print(dataset.size().getInfo())

214


In [ ]:
red_band='B4'
green_band='B3'
blue_band='B2'
nir_band='B8'
swir_band1='B11'
swir_band2='B12'


In [ ]:
count = dataset.size().getInfo()
bands = [red_band,green_band,blue_band,nir_band,swir_band1,swir_band2]
for i in range(count):
  img = dataset.toList(dataset.size()).get(i)
  img = ee.Image(img)
  date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
  task = ee.batch.Export.image.toDrive(
    image=img.select(bands),  # example bands
    description=f"SyedanWalan_Sentinel_{date}",       # Task name
    folder="GEE",           # Drive folder
    fileNamePrefix=f"SyedanWalan_Sentinel_{date}",    # File name
    region=roi,
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e13
  )
  task.start()
  print(f"Exporting {date}...")
  print(date)

Exporting 2018-01-01...
2018-01-01
Exporting 2018-01-06...
2018-01-06
Exporting 2018-01-08...
2018-01-08
Exporting 2018-01-13...
2018-01-13
Exporting 2018-01-16...
2018-01-16
Exporting 2018-01-21...
2018-01-21
Exporting 2018-02-02...
2018-02-02
Exporting 2018-03-07...
2018-03-07
Exporting 2018-03-17...
2018-03-17
Exporting 2018-03-27...
2018-03-27
Exporting 2018-03-29...
2018-03-29
Exporting 2018-04-06...
2018-04-06
Exporting 2018-04-08...
2018-04-08
Exporting 2018-04-26...
2018-04-26
Exporting 2018-05-23...
2018-05-23
Exporting 2018-05-28...
2018-05-28
Exporting 2018-06-02...
2018-06-02
Exporting 2018-06-05...
2018-06-05
Exporting 2018-08-01...
2018-08-01
Exporting 2018-10-05...
2018-10-05
Exporting 2018-10-20...
2018-10-20
Exporting 2018-10-28...
2018-10-28
Exporting 2018-11-07...
2018-11-07
Exporting 2018-11-19...
2018-11-19
Exporting 2018-11-24...
2018-11-24
Exporting 2018-12-14...
2018-12-14
Exporting 2018-12-17...
2018-12-17
Exporting 2018-12-19...
2018-12-19
Exporting 2018-12-22